In [ ]:
# ses-nota 8. TUR: 7. turun tarifi, büyüyen gerçek veri + kanun/klarnet sentetiği; sınavlar ham ve temizlenmiş -- TEK HÜCRE. A100.
# 7. turun ağırlıklarından devam eder. Yenilikler:
#  - gerçek kayıtlar ince-ayar-6'da: şarkı söylenen kayıtların sesi çıkarılmış hâli (-ayrik), şarkıcı + tek saz videoları,
#    eşleştirmede ikinci bakış ve parça kuralı, nota havuzu düzeltmesi;
#  - sentetiğe ornek2 eklendi: solo klarnet (düzeltilmiş örneklerle) ve kanunun dolgu, tril, oktav vuruşu, oktav çifti;
#  - her sınav ham ve temizlenmiş (model/temizle.py: tekrar vuruşları, çok kısa notalar; solo kanunda daha sıkı kural)
#    olarak puanlanır; dürüst sınav ayrıca güvenilir kayıtlarla (A+B) ayrı verilir.
# Sonunda 7. turun ağırlıkları da aynı sınavlarda ölçülür.
AD = 'tur8-v1'        # koşu adı: runs/ic/<tarih>-<AD>/
BASLANGIC = None      # None: 7. turun son koşusunun son.pt'si · ya da bir .pt yolu
ADIM = 20000
DEVAM = None          # oturum kesilirse: 'son' (bu 8. tur koşusunun son checkpoint'i)
KAPAT = True         # bitince Drive'ı kaydedip çalışma zamanını kapat (kota harcamasın)
A100_SART = True     # A100 yoksa eğitime başlamadan dur · False: yine de koş
# ----------------------------------------------------------------------------------------------------
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU yok'
print('GPU:', GPU)
if A100_SART and 'A100' not in GPU:
    raise SystemExit(f'A100 değil ({GPU}): Runtime → Change runtime type → A100 GPU seç, yeniden çalıştır. '
                     'A100 olmadan koşmak istersen A100_SART = False yap.')
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, zipfile, json, glob
from pathlib import Path
DRIVE = '/content/drive/MyDrive/ses_nota'
KOD = '/content/ses-nota'
shutil.rmtree(KOD, ignore_errors=True)
zipfile.ZipFile(f'{DRIVE}/kod/kod.zip').extractall(KOD)
os.environ['SES_NOTA_HOMR'] = f'{KOD}/homr_kopya'
!pip -q install librosa soundfile musicxml==1.4 mir_eval
sys.path[:0] = [f'{KOD}/model', f'{KOD}/model/colab']
import defter
KAYIT = f'{DRIVE}/veri/ic/sinav-kayit-6'
BOLUM = f'{DRIVE}/veri/ic/ince-ayar-6/bolum.json'
_kapat = KAPAT


def temizle(sinav_dir, symbtr):
    """Ham tahminlerin yanına temizlenmiş kopyaları ve puanlarını yazar (tahmin_temiz/, ozet_temiz.json)."""
    for s in ('kayit', 'tanbur'):
        d = Path(sinav_dir) / s
        if d.is_dir():
            cmd = [sys.executable, 'temizle.py', '--sinav', str(d), '--kanun-manifest', f'{KAYIT}/manifest.json']
            defter.sh(cmd + (['--symbtr-txt', symbtr] if symbtr else []), cwd=f'{KOD}/model')


def durust(run, alt):
    """Dürüst sınav (sinav__ grupları), nota ağırlıklı SER: hepsi ve yalnız güvenilir (A+B) kayıtlar; ham ve temiz."""
    sinif = {e['anahtar']: (e.get('guvenilirlik') or {}).get('sinif') for e in
             json.loads(Path(f'{KAYIT}/manifest.json').read_text(encoding='utf-8'))['eserler']}
    out = {}
    for ek in ('', '_temiz'):
        top = {'hepsi': [0, 0.0], 'A+B': [0, 0.0]}
        for g in ('sinav__symbtr_test', 'sinav__symbtr_yok'):
            p = run / alt / 'kayit' / f'rapor_{g}{ek}.json'
            if not p.exists():
                continue
            for f in json.loads(p.read_text(encoding='utf-8'))['dosyalar']:
                n, s = f['sonuc']['dizi']['ref_nota'], f['sonuc']['dizi']['ser']
                for k in ('hepsi',) + (('A+B',) if sinif.get(f['ad']) in ('A', 'B') else ()):
                    top[k][0] += n
                    top[k][1] += n * s
        out['ham' if not ek else 'temiz'] = {k: round(v[1] / v[0], 4) if v[0] else None for k, v in top.items()}
    return out


try:
    KOK = f'{DRIVE}/runs/ic'
    runs = sorted((p for p in Path(KOK).glob('*') if (p / 'ayar.json').exists()), key=lambda p: p.stat().st_mtime)
    if BASLANGIC is None:
        onceki = [p for p in runs if 'tur7' in p.name and (p / 'son.pt').exists()]
        BASLANGIC = str(onceki[-1] / 'son.pt')
    print('başlangıç ağırlıkları:', BASLANGIC)
    devam = DEVAM
    if DEVAM == 'son':
        ince = [p for p in runs if 'tur8' in p.name]
        pts = sorted(glob.glob(f'{ince[-1]}/kontrol_*.pt')) if ince else []
        devam = pts[-1] if pts else None
        print('devam:', devam)
    print(defter.copy_shards(f'{DRIVE}/veri', '/content/veri',
                             ['acik/train', 'ic/ddsp/train', 'ic/ornek/train', 'acik3/train', 'ic/duet/train',
                              'ic/ornek2/train', 'acik/val', 'acik3/val', 'ic/gercek6/train', 'ic/gercek6/val']))
    # the real set is new on Drive: stop before training if Drive has not synced all of it yet
    beklenen = json.loads(Path(f'{DRIVE}/veri/ic/ince-ayar-6/index.json').read_text(encoding='utf-8'))['bolumler']
    for b in ('train', 'val'):
        n = len(list(Path(f'/content/veri/ic/gercek6/{b}').glob('*.tar')))
        if n < beklenen[b]['shard']:
            raise SystemExit(f'gercek6/{b}: {n}/{beklenen[b]["shard"]} shard. Drive henüz eşitlenmemiş: '
                             'biraz bekleyip yeniden çalıştır.')
    cfg = json.loads(Path(f'{KOD}/model/ayarlar/tur8.json').read_text(encoding='utf-8'))
    cfg['ad'] = AD
    cfg['egitim']['baslangic'] = BASLANGIC
    cfg['egitim']['adim'] = ADIM
    cfg['veri']['isci'] = min(8, os.cpu_count() or 2)
    kod = defter.train(KOD, cfg, f'/content/ayar_{AD}.json', resume=devam)
    run = defter.latest_run(KOK)
    print('koşu:', run, '| çıkış kodu:', kod)
    if kod == 0:
        KONTROL = str(run / 'son.pt')
        symbtr = f'{DRIVE}/veri/acik/symbtr_txt'
        symbtr = symbtr if os.path.isdir(symbtr) else None
        ic = dict(tanbur_dir=f'{DRIVE}/veri/ic/sinav-tanbur', ic_out_root=f'{DRIVE}/runs/ic/sinav', symbtr_txt=symbtr)
        sonuc = defter.evaluate(KOD, KONTROL, ['sentetik', 'tanbur', 'kayit'], str(run / 'sinav'),
                                synthetic_shards=[f'{DRIVE}/veri/acik/test'], karar='verilen',
                                kayit_dir=KAYIT, ince_ayar_bolum=BOLUM, **ic)
        print(json.dumps(sonuc, ensure_ascii=False, indent=1)[:6000])
        onceki = defter.evaluate(KOD, BASLANGIC, ['tanbur', 'kayit'], str(run / 'sinav_baslangic'), karar='verilen',
                                 kayit_dir=KAYIT, ince_ayar_bolum=BOLUM, **ic)
        # guitar / piano accompanied ney: evaluation only, never trained on; grouped by the SymbTr split
        eslikli = {ad: defter.evaluate(KOD, pt, ['kayit'], str(run / ad), karar='verilen',
                                       kayit_dir=f'{KAYIT}/eslikli', **ic)
                   for ad, pt in (('sinav_eslikli', KONTROL), ('sinav_eslikli_baslangic', BASLANGIC))}
        for alt in ('sinav', 'sinav_baslangic', 'sinav_eslikli', 'sinav_eslikli_baslangic'):
            temizle(run / alt, symbtr)

        def ser(r):
            return {f'{s}/{g}': (v.get('dosya'), v.get('dizi.ser')) for s, x in r.items()
                    for g, v in (x.get('gruplar') or {}).items() if not g.startswith(('ince_ayar__', 'kullanilmaz__'))}

        def temiz(alt):
            return {s: json.loads((run / alt / s / 'ozet_temiz.json').read_text(encoding='utf-8'))
                    for s in ('kayit', 'tanbur') if (run / alt / s / 'ozet_temiz.json').exists()}

        yeni, eski = ser(sonuc), ser(onceki)
        yeni.update({'eslikli/' + k: v for k, v in ser(eslikli['sinav_eslikli']).items()})
        eski.update({'eslikli/' + k: v for k, v in ser(eslikli['sinav_eslikli_baslangic']).items()})
        yeni_t, eski_t = ser(temiz('sinav')), ser(temiz('sinav_baslangic'))
        yeni_t.update({'eslikli/' + k: v for k, v in ser(temiz('sinav_eslikli')).items()})
        eski_t.update({'eslikli/' + k: v for k, v in ser(temiz('sinav_eslikli_baslangic')).items()})
        print('SER (düşük iyi)            dosya   7. tur   8. tur | temiz: 7. tur   8. tur')
        for g in sorted(yeni):
            print(f'{g:34s} {yeni[g][0]!s:>4}   {(eski.get(g) or (None, None))[1]!s:>6}   {yeni[g][1]!s:>6} |'
                  f'        {(eski_t.get(g) or (None, None))[1]!s:>6}   {(yeni_t.get(g) or (None, None))[1]!s:>6}')
        print('dürüst sınav, nota ağırlıklı SER: 7. tur', durust(run, 'sinav_baslangic'), '| 8. tur', durust(run, 'sinav'))
        icra = defter.evaluate(KOD, KONTROL, ['sentetik'], str(run / 'sinav_icra'), karar='verilen', kip='icra',
                               synthetic_shards=[f'{DRIVE}/veri/acik3/val'], symbtr_txt=symbtr)
        defter.evaluate(KOD, KONTROL, ['kayit'], str(run / 'sinav_icra'), karar='verilen', kip='icra',
                        kayit_dir=KAYIT, ince_ayar_bolum=BOLUM, **ic)
        print('çalındığı gibi nota (kip_icra), süslemeli sentetik doğrulama, çarpmalar sayılır:',
              json.dumps((icra.get('sentetik') or {}).get('gruplar'), ensure_ascii=False)[:600])
        print('albüm kayıtlarının çalındığı gibi notaları (MusicXML):', run / 'sinav_icra')
except KeyboardInterrupt:  # stopped by hand: keep the runtime
    _kapat = False
    raise
finally:
    print('BİTTİ.')
    if _kapat:
        print('Drive kaydediliyor, çalışma zamanı kapatılıyor (KAPAT = True)...')
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()
    else:
        print('Runtime → Disconnect and delete runtime (A100 boşta da kota harcar).')